# Expanded multi-rate $1T smoke test

Uses completed expanded source partitions, Polars streaming, pre-2024 labels, and the existing annual anchored-HITS evaluator. Requires the recorded local artifacts. No holder-exit records exist in this subset. Sparse historical inputs use disclosure dates; congressional/insider targets use transaction dates.

In [ ]:
import json,hashlib
from pathlib import Path
from datetime import datetime
import polars as pl
from quant_warehouse import Warehouse
from quant_warehouse.platforms.data_providers.fmp.feature_engineering.event_observations import issuer_event_observations,CHANNELS
from quant_orchestrator.research_tools.multirate_supervision import SUPERVISED_ONLY_FAMILIES
root=Path('artifacts/multirate_recovery/100B').resolve();base=root/'corpus_inference_2026_v6';wide=root/'corpus_expanded_v9';out=root.parent/'1T/corpus_expanded_v9';out.mkdir(exist_ok=True)
w=Warehouse();tax=pl.read_csv(root.parent/'1T/corpus_long_v2/taxonomy.csv');tax.write_csv(out/'taxonomy.csv');symbols=sorted(tax['underlying_symbol'].unique().to_list())
manifest=json.loads((base/'manifest.json').read_text());coverage=json.loads((wide/'family_coverage.json').read_text());coverage['records']=[r for r in coverage['records'] if r['symbol'] in tax['symbol'].to_list()];records=coverage['records'];seen={r['family'] for r in records}
assert seen==set(coverage['expected']),set(coverage['expected'])-seen
coverage['observed']=sorted(seen);(out/'family_coverage.json').write_text(json.dumps(coverage,indent=2))
features=sorted(set(manifest['feature_families'])|{r['family']+'.'+c for r in records for c in r['fields']})
for rate in ['annual','quarterly','daily']:
 files=[wide/'partitions'/f'merged_{s}_{rate}.parquet' for s in tax['symbol']]
 files=[f for f in files if f.exists()]
 pl.scan_parquet(files).select('symbol','date',*['value__'+f for f in features]).sink_parquet(out/f'{rate}.parquet',row_group_size=8192)
 print('numeric complete',rate,flush=True)
# Preserve original outcome labels only; rebuild disclosure-sensitive inputs.
sparse=[pl.scan_parquet(base/'sparse_events.parquet').filter(pl.col('target_family').is_in(list(SUPERVISED_ONLY_FAMILIES)) & pl.col('symbol').is_in(tax['symbol'].to_list()))]
for symbol in symbols:
 for section,f in issuer_event_observations(w,symbol):
  if not f.is_empty():sparse.append(f.lazy())
from quant_warehouse.research_tools.fund_activity import build_institutional_activity_events
for symbol in symbols:
 frame=w.read_fundamentals(symbol,section='ownership_institutional',start='1900-01-01',end='2026-09-09')
 if frame.is_empty():continue
 frame=frame.with_columns(pl.lit(symbol).alias('symbol'))
 events=build_institutional_activity_events(frame)
 if not events.is_empty():
  sparse.append(events.select('symbol','date','event_date','target_family',pl.col('signal_value').cast(pl.Float32),*[pl.lit(None,dtype=pl.Float32).alias(c) for c in CHANNELS[1:]]).lazy())
# M&A and fund/holder event exports retain their stored availability, never Oracle/HITS inputs.
for source,allowed in [(root.parent.parent/'multi-rate-mtl/inputs/100B/target_events.parquet',['equity.corporate.merger_acquisition']),
                       (root.parent.parent/'multi-rate-mtl/inputs/100B/fund_activity_events_with_holder.parquet',None)]:
 f=pl.scan_parquet(source).filter(pl.col('symbol').is_in(symbols))
 if allowed:f=f.filter(pl.col('target_family').is_in(allowed))
 schema=f.collect_schema()
 dates=[]
 for c in ['date','reported_date']:
  if c in schema:
   e=pl.col(c)
   if isinstance(schema[c],pl.Datetime) and schema[c].time_zone:e=e.dt.convert_time_zone('UTC').dt.replace_time_zone(None)
   dates.append(e.cast(pl.Datetime('ns')))
 event=pl.col('event_date')
 if isinstance(schema['event_date'],pl.Datetime) and schema['event_date'].time_zone:event=event.dt.convert_time_zone('UTC').dt.replace_time_zone(None)
 f=f.select('symbol',pl.max_horizontal(dates).alias('date'),event.cast(pl.Datetime('ns')).alias('event_date'),'target_family',*[pl.col(c).cast(pl.Float32) if c in schema else pl.lit(None,dtype=pl.Float32).alias(c) for c in CHANNELS])
 sparse.append(f)
combined=pl.concat(sparse,how='diagonal_relaxed').with_columns(pl.col('date','event_date').cast(pl.Datetime('ns'))).filter(pl.col('date')<=datetime(2026,9,9))
combined.sink_parquet(out/'sparse_events.parquet',row_group_size=8192)
sparse_coverage=pl.scan_parquet(out/'sparse_events.parquet').group_by('target_family').agg(pl.len().alias('rows'),pl.col('date').min().alias('first'),pl.col('date').max().alias('last'),(pl.col('date')<datetime(2024,1,1)).sum().alias('pre2024')).collect(engine='streaming')
sparse_coverage.write_csv(out/'sparse_coverage.csv')
targets=sorted(sparse_coverage['target_family'].to_list())
assert len(targets)==16, targets  # No holder exit events are recorded for the 1T subset.
new={**manifest,'feature_families':features,'target_families':targets,'start':'1900-01-01','end':'2026-09-09','build_mode':'expanded_bounded_polars','expanded_source_manifest':'family_coverage.json','min_market_cap':1000000000000,'symbols':tax['symbol'].to_list(),'input_sha256':{}}
for name in ['annual.parquet','quarterly.parquet','daily.parquet','sparse_events.parquet','taxonomy.csv']:
 with (out/name).open('rb') as file:new['input_sha256'][name]=hashlib.file_digest(file,'sha256').hexdigest()
(out/'manifest.json').write_text(json.dumps(new,indent=2));(out/'build_status.json').write_text(json.dumps(dict(stage='complete',feature_families=len(seen),individual_features=len(features),sparse_families=len(targets))))


In [ ]:
"""Fresh model training with a blocking epoch backtest monitor."""
import json,subprocess,sys,time,traceback
from pathlib import Path
root=Path('artifacts/multirate_recovery/1T').resolve();repo=Path.cwd();output=root/'train_expanded_smoke_v9';output.mkdir(exist_ok=False)
evaluation=output/'epoch_validation_2024_2026';evaluation.mkdir()
base=json.loads((root.parent/'100B/training_command_fresh_v8.json').read_text())
for flag,value in {'--corpus':str(root/'corpus_expanded_v9'),'--epochs':'1','--patience':'1','--batch-size':'32','--prediction-end-date':'2026-09-09'}.items():base[base.index(flag)+1]=value
del base[base.index('--epoch-evaluation-dir'):base.index('--epoch-evaluation-dir')+2]
assert '--checkpoint' not in base and '--resume-training' not in base
base[base.index('--output-dir')+1]=str(output)
base += ['--epoch-evaluation-dir',str(evaluation)]
command_file=root/'training_command_expanded_smoke_v9.json';command_file.write_text(json.dumps(base,indent=2))
(output/'launch_manifest.json').write_text(json.dumps(dict(initialization='fresh random model weights and fresh optimizer',checkpoint=None,resume=False,epochs=1,min_market_cap=1000000000000,training_cutoff='2024-01-01',backtest_periods=['2024','2025','2026_YTD_through_2026-09-09'],blocking_epoch_backtests=True,command=base),indent=2))
def status(stage,**kw):
 p=root/'run_status.tmp';p.write_text(json.dumps(dict(stage=stage,supervisor_pid=__import__('os').getpid(),output=str(output),from_scratch=True,synchronous_epoch_backtests=True,**kw),indent=2));p.replace(root/'run_status.json')
trainer=monitor=None
try:
 with (root/'training_expanded_smoke_v9.log').open('w') as training_log,(root/'epoch_validation_expanded_smoke_v9.log').open('w') as evaluation_log:
  trainer=subprocess.Popen(base+['--skip-predictions'],stdout=training_log,stderr=subprocess.STDOUT)
  monitor=subprocess.Popen([sys.executable,str(repo/'scripts/monitor_multirate_epochs.py'),'--command-file',str(command_file),'--training-log',str(root/'training_expanded_smoke_v9.log'),'--output-dir',str(evaluation),'--validation-start','2024-01-02','--validation-end','2026-09-09','--inference-corpus',str(root/'corpus_expanded_v9'),'--backtest-by-year','--training-pid',str(trainer.pid),'--backtest-anchored-hits'],stdout=evaluation_log,stderr=subprocess.STDOUT)
  status('fresh_training_with_epoch_backtests',pid=trainer.pid,monitor_pid=monitor.pid)
  while trainer.poll() is None:
   if monitor.poll() is not None:raise RuntimeError('Epoch monitor exited while training was active')
   time.sleep(5)
  if trainer.returncode:raise RuntimeError('Training failed; inspect training_expanded_smoke_v9.log')
  if monitor.wait():raise RuntimeError('Epoch monitor failed')
 status('fresh_training_and_all_epoch_backtests_complete')
except Exception as exc:
 (evaluation/'failure.json').write_text(json.dumps({'error':str(exc)}))
 for child in (trainer,monitor):
  if child is not None and child.poll() is None:child.terminate()
 status('failed',error=str(exc));traceback.print_exc();raise
